In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler, FunctionTransformer, SplineTransformer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.model_selection import StratifiedKFold, cross_val_score

In [2]:
# amaga el warning q surt cada cop q evaluem amb el model q demana al campus
import warnings

warnings.filterwarnings(
    "ignore",
    message="'penalty' was deprecated in version 1.8",
    category=FutureWarning
)

## EVALUACIÓ AMB LOGISTIC REGRESSOR

De momento solo provamos con train.csv asi el "test" que nos salga sera como "evaluation"

In [3]:
#cargar data
train = pd.read_csv('./train.csv')

In [4]:
print(f"Files originals: {len(train)}")

# Directament amb dropna indicant el mínim de valors no nuls que necessites per fila (per exemple, si tens 26 columnes en total, exigir un mínim de 22 columnes amb dades)
train = train.dropna(thresh=22).copy()


print(f"Files després de netejar: {len(train)}")

Files originals: 2400
Files després de netejar: 2392


remove outliers:

In [5]:
def clip_outliers(df, column):
    df = df.copy()

    Q1 = df[column].quantile(0.05)
    Q3 = df[column].quantile(0.95)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df[column] = df[column].clip(
        lower=lower_bound,
        upper=upper_bound
    )

    return df

outliers_columns =  [
    'creatinine_mg_dl',
    'length_of_stay_days',
    'sodium_mmol_l',
    'glucose_mg_dl'
]
for column in outliers_columns:
    train = clip_outliers(train, column)

In [91]:
X = train.drop(columns=["id","label"])
y = train["label"]

feature engineering:

In [92]:
X["mean_arterial_pressure"] = (
    X["systolic_bp"] + 2 * X["diastolic_bp"]
) / 3

X.drop(
    columns=["systolic_bp", "diastolic_bp"],
    inplace=True
)

X["bmi_age_interaction"] = (
        X["bmi"] * X["age"]
    )

X.drop(
    columns=["age"],
    inplace=True
)


In [93]:
binary_columns = [
    "diabetes",
    "hypertension",
    "chronic_kidney_disease",
    "readmitted_30d"
]

for column in binary_columns:
    X[column] = X[column].map({"No": 0, "Yes": 1})

In [94]:
for column in binary_columns:
    print(column, X[column].unique())

diabetes [0 1]
hypertension [0 1]
chronic_kidney_disease [0 1]
readmitted_30d [0 1]


separete train / test : (fent cross-validation no cal)

In [95]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

divide the different variables:

In [96]:
# Variables numéricas discretas
discrete_features = [
    #"age",
    #"systolic_bp",
    #"diastolic_bp",
    "heart_rate",
    "respiratory_rate",
    "spo2",
    "num_medications",
    "prior_visits_12m",
    "mean_arterial_pressure",
]

binary_features = [
    "diabetes",
    "hypertension",
    "chronic_kidney_disease",
    "readmitted_30d"
]

# Variables numéricas continuas
continuous_features = [
    "bmi",
    "temperature_c",
    "glucose_mg_dl",
    "creatinine_mg_dl",
    "hemoglobin_g_dl",
    "sodium_mmol_l",
    "length_of_stay_days",
    "bmi_age_interaction",
]

# Variables categóricas
cat_features = [
    "sex",
    "smoking_status",
    "encounter_type",
    "primary_diagnosis",
    "discharge_destination",
]

In [97]:
# Pipeline para discretas
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

discrete_pipeline = Pipeline([
    #("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    #("imputer", KNNImputer(n_neighbors=10)), # Omple els NaN mirant els 5 veïns més propers
    ("imputer",
        IterativeImputer(
            estimator=BayesianRidge(),
            max_iter=15,
            initial_strategy="median",
            add_indicator=True,
            random_state=42
        )),
    ("scaler", StandardScaler()),
])

# Pipeline para continuas
continuous_pipeline = Pipeline([
    #("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    #("imputer", KNNImputer(n_neighbors=10)), # Omple els NaN mirant els 5 veïns més propers
    ("imputer",
            IterativeImputer(
                estimator=BayesianRidge(),
                max_iter=15,
                initial_strategy="median",
                add_indicator=True,
                random_state=42
            )),
    ("scaler", RobustScaler()),
])

# Pipeline para categóricas
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

In [98]:
# Keep length_of_stay_days out of the regular continuous pipeline
other_continuous_features = [
    col for col in continuous_features
    if col != "length_of_stay_days"
]


length_of_stay_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("spline", SplineTransformer(
        n_knots=4,
        degree=2,
        include_bias=False
    )),
    ("scaler", StandardScaler()),
])

spline_preprocessor = ColumnTransformer([
    ("discrete", discrete_pipeline, discrete_features),
    ("continuous", continuous_pipeline, other_continuous_features),
    ("length_of_stay_spline", length_of_stay_pipeline,
     ["length_of_stay_days"]),
    ("categorical", cat_pipeline, cat_features),
    ("binary", "passthrough", binary_features),
])

model_spline = Pipeline([
    ("preprocessor", spline_preprocessor),
    ("classifier", LogisticRegression(
        penalty=None,
        max_iter=5000,
        random_state=42
    )),
])

In [99]:
preprocessor = ColumnTransformer([
    ("discrete", discrete_pipeline, discrete_features),
    ("continuous", continuous_pipeline, other_continuous_features),
        ("length_of_stay_spline", length_of_stay_pipeline,
         ["length_of_stay_days"]),
    ("categorical", cat_pipeline, cat_features),
    ("binary", "passthrough", binary_features),
])

el modelo:

In [100]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=5000,
        penalty=None,
        random_state=42
    ))
])

In [101]:
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Train accuracy:", accuracy_score(y_train, model.predict(X_train)))
print("Validation accuracy:", accuracy_score(y_test, y_pred))


Train accuracy: 0.810768426555149
Validation accuracy: 0.8121085594989561


using cross-validation with k-fold for a better accuracy score:

In [102]:
cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42
)
scores = cross_val_score(
    model_spline,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)

print("Accuracy for each fold:", scores.round(4))
print("Mean validation accuracy:", round(scores.mean(), 4))
print("Variation between folds:", round(scores.std(), 4))

Accuracy for each fold: [0.8246 0.7912 0.7322 0.7845 0.8243 0.7912 0.7808 0.7615 0.8117 0.7762
 0.7704 0.7954 0.7762 0.8033 0.7866 0.7683 0.785  0.7992 0.7762 0.7887
 0.8017 0.833  0.7824 0.7741 0.7908]
Mean validation accuracy: 0.7884
Variation between folds: 0.0211


In [103]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score
)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nMatriu de confusió:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.8121085594989561

Matriu de confusió:
[[ 82  11   7  20]
 [ 11  96   8   4]
 [  5   7 105   3]
 [  8   5   1 106]]

Classification report:
              precision    recall  f1-score   support

     Model_A       0.77      0.68      0.73       120
     Model_B       0.81      0.81      0.81       119
     Model_C       0.87      0.88      0.87       120
     Model_D       0.80      0.88      0.84       120

    accuracy                           0.81       479
   macro avg       0.81      0.81      0.81       479
weighted avg       0.81      0.81      0.81       479



# TO SUMBIT

Aqui si que utilitzarem el test.csv. (Millor basar-nos només en el de dalt)

In [140]:
train = pd.read_csv('./train.csv')
test = pd.read_csv('./test.csv')

remove outliers:

In [141]:
train = train.dropna(thresh=22).copy()

In [142]:
for column in outliers_columns:
    train = clip_outliers(train, column)

feature engineering:

In [143]:
train["mean_arterial_pressure"] = (
    train["systolic_bp"] + 2 * train["diastolic_bp"]
) / 3

train.drop(
    columns=["systolic_bp", "diastolic_bp"],
    inplace=True
)

train["bmi_age_interaction"] = (
        train["bmi"] * train["age"]
    )

train.drop(
    columns=["age"],
    inplace=True
)

test["mean_arterial_pressure"] = (
    test["systolic_bp"] + 2 * test["diastolic_bp"]
) / 3

test.drop(
    columns=["systolic_bp", "diastolic_bp"],
    inplace=True
)

test["bmi_age_interaction"] = (
        test["bmi"] * test["age"]
    )

test.drop(
    columns=["age"],
    inplace=True
)

In [144]:
binary_columns = [
    "diabetes",
    "hypertension",
    "chronic_kidney_disease",
    "readmitted_30d"
]

for column in binary_columns:
    train[column] = train[column].map({"No": 0, "Yes": 1})
    test[column] = test[column].map({"No": 0, "Yes": 1})

In [145]:
for column in binary_columns:
    print(column, train[column].unique())

diabetes [0 1]
hypertension [0 1]
chronic_kidney_disease [0 1]
readmitted_30d [0 1]


Defining X and y:

In [146]:

X_train = train.drop(columns=["label","id"])
y_train = train["label"]

test_ids = test["id"]
X_test = test.drop(columns=["id"])

In [147]:
# Pipeline para discretas
discrete_pipeline = Pipeline([
    #("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    #("imputer", KNNImputer(n_neighbors=10)), # Omple els NaN mirant els 5 veïns més propers
    ("imputer",
        IterativeImputer(
            estimator=BayesianRidge(),
            max_iter=15,
            initial_strategy="median",
            add_indicator=True,
            random_state=42
        )),
    ("scaler", StandardScaler()),
])

# Pipeline para continuas
continuous_pipeline = Pipeline([
    #("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    #("imputer", KNNImputer(n_neighbors=10)), # Omple els NaN mirant els 5 veïns més propers
    ("imputer",
        IterativeImputer(
            estimator=BayesianRidge(),
            max_iter=15,
            initial_strategy="median",
            add_indicator=True,
            random_state=42
        )),
    ("scaler", RobustScaler()),
])

# Pipeline para categóricas
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

In [148]:
#intentant spline per length_of_stay_days

# Keep length_of_stay_days out of the regular continuous pipeline
other_continuous_features = [
    col for col in continuous_features
    if col != "length_of_stay_days"
]


length_of_stay_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("spline", SplineTransformer(
        n_knots=4,
        degree=2,
        include_bias=False
    )),
    ("scaler", StandardScaler()),
])

spline_preprocessor = ColumnTransformer([
    ("discrete", discrete_pipeline, discrete_features),
    ("continuous", continuous_pipeline, other_continuous_features),
    ("length_of_stay_spline", length_of_stay_pipeline,
     ["length_of_stay_days"]),
    ("categorical", cat_pipeline, cat_features),
    ("binary", "passthrough", binary_features),
])

model_spline = Pipeline([
    ("preprocessor", spline_preprocessor),
    ("classifier", LogisticRegression(
        penalty=None,
        max_iter=5000,
        random_state=42
    )),
])

In [149]:
preprocessor = ColumnTransformer([
    ("discrete", discrete_pipeline, discrete_features),
    ("continuous", continuous_pipeline, continuous_features),
    ("categorical", cat_pipeline, cat_features),
])

In [150]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=4000,
        penalty=None,
        random_state=42
    ))
])

In [151]:
#Ja tenim totes les configuracions d'abans aixi que nomes ens cal tornar a evaluar el model trobat

model_spline.fit(X_train, y_train)

predictions = model_spline.predict(X_test)

# Create the submission DataFrame
submission = pd.DataFrame({
    "id": test_ids,
    "label": predictions
})

# Save to CSV
submission.to_csv("submission_c.csv", index=False)

print("Train accuracy:", accuracy_score(y_train, model_spline.predict(X_train)))
print(submission.head())
print("Submission shape:", submission.shape)

Train accuracy: 0.8173076923076923
                                 id    label
0  0102f98c5fd7ed84754c58fe0d7f013f  Model_D
1  e2fa3e497f5bdb8c79748c7c0611fad0  Model_B
2  818a6ec782d097abc1ce67c42299d06a  Model_A
3  54e46a096cb47b5499a986477be83061  Model_B
4  5da9e7a746719fc3950c6e579c82bfa1  Model_A
Submission shape: (1600, 2)
